# 18. 저장된 DB 청크로 질문 검색하기

**순서:** 질문 입력 → 질문만 임베딩 → DB의 11개 청크 검색 → 연결 원문·각주·그림 보기.

- [프로젝트 적용] 기존 한국어 로컬 모델 `jhgan/ko-sroberta-multitask-mrl`과 SQL Mapper를 사용합니다. 문서와 같은 모델 버전·768차원으로 질문을 계산합니다.
- [프로젝트 추가] 개인 표본 run_id로 검색 범위를 제한하고, 같은 원문 묶음의 중복을 줄이며 필수 각주와 그림 설명을 연결합니다.
- 현재 검색 대상은 PDF 43~45쪽의 부모 4개·청크 11개입니다. 전체 PDF 검색이나 생성 답변은 아직 아닙니다.
- DB는 읽기 전용입니다. 이 노트북은 저장·업로드·문서 임베딩 재계산을 하지 않습니다.
- 프로젝트 `.venv` 커널을 선택하고 위에서 아래로 셀을 실행하세요. 실행 결과는 아직 이 노트북에 저장하지 않았습니다.


In [ ]:
from pathlib import Path
import sys

# [프로젝트 적용] notebooks에서 실행해도 프로젝트의 src 폴더를 찾아 Python 코드를 불러옵니다.
project_folder = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                       if (folder / "src/car_search_rag/zzong_santafe_lag").is_dir()), None)
if project_folder is None:
    raise FileNotFoundError("프로젝트 폴더에서 이 노트북을 열어 주세요.")
sys.path.insert(0, str(project_folder / "src"))

# 서비스는 여러 작은 작업을 정해진 순서로 호출하는 클래스입니다.
from car_search_rag.zzong_santafe_lag.db_search_service import DbSampleSearchService
from car_search_rag.zzong_santafe_lag.manual import print_db_result


## 1. 저장 버전과 로컬 모델 준비

`prepare()`는 DB 표본과 모델 설정을 확인하고 로컬 모델을 한 번 읽습니다. 같은 커널에서는 아래 질문을 바꿔도 모델을 재사용합니다. 모델 파일이 없으면 다운로드하지 않고 멈춥니다. DB 연결 비밀번호는 출력하지 마세요.


In [ ]:
# [프로젝트 추가] 개인 .env의 ZZONG_DB_URL로 읽기 전용 조회를 준비합니다.
service = DbSampleSearchService()
service.prepare(progress=print)

print("저장된 부모·청크:", service.counts)
print("사용 모델:", service.embedder.model_name)
print("모델 버전:", service.embedder.model_revision)
print("질문 입력 한도:", service.token_counter.budget, "토큰")


## 2. 질문 한 개 검색

처음에는 아래 차대번호 질문을 그대로 실행하세요. 그다음 `question`만 바꾸고 **이 셀부터 다시 실행**하면 됩니다.

다른 예:
- 엔진 오일의 용량과 추천 사양을 알려줘.
- 엔진 오일의 SAE 점도는 무엇인가요?

`top_k=1`은 가장 높은 **근거 묶음** 한 개를 보여 줍니다. 여러 청크가 같은 원문을 가리킬 때 중복을 줄인 결과입니다. 유사도는 순위 점수이며 정답 확률이 아닙니다. 표본에 없는 질문에도 후보가 나올 수 있습니다.


In [ ]:
# [프로젝트 적용] 질문만 768개 숫자로 바꾸고 DB의 기존 문서 벡터와 비교합니다.
question = "차대번호는 어디에서 확인하나요?"
result = service.search(question, top_k=1, progress=print)

# full_text=True로 연결된 원문과 각주 전체를 보여 줍니다. 새 답변을 생성하지 않습니다.
print_db_result(result, full_text=True)


## 3. 출처와 각주 연결 확인

검색된 작은 청크만으로 답하면 중요한 주의사항이 빠질 수 있습니다. 그래서 `context_records`에는 부모 글과 필수 각주를 함께 담습니다. 오일 용량 질문은 43쪽의 표·각주 두 기록이 연결됩니다. PDF 쪽수와 매뉴얼 인쇄 쪽수는 서로 다릅니다.


In [ ]:
# [프로젝트 추가] 검색 결과 딕셔너리에서 출처·각주·그림 개수를 꺼내 봅니다.
for hit in result["candidates"]:
    print("검색된 제목:", hit["title"])
    print("필수 문맥을 포함한 기록 수:", len(hit["context_records"]))
    for record in hit["context_records"]:
        print("원문 ID:", record["record_id"])
        print("PDF 쪽:", record["source_pages"], "| 매뉴얼 쪽:", record["manual_page_number"])
        print("검토 상태:", record["verification_status"])
    print("연결된 그림 수:", len(hit["image_parts"]))


## 4. 관련 그림 표시

DB에는 그림 파일 자체 대신 위치와 설명이 있습니다. 아래 셀은 공개 Storage 주소의 그림을 화면에 표시합니다. 그림을 새로 업로드하거나 그림 자체를 임베딩하지 않습니다. 오일 용량 표는 글로 구조화한 표라 연결 그림이 없고, 점도는 2개, 차대번호는 1개입니다.


In [ ]:
from IPython.display import Image, display

# [프로젝트 추가] 이번 검색에 연결된 그림만 표시하고 같은 파일은 한 번만 보여 줍니다.
shown_urls = set()
for hit in result["candidates"]:
    for image in hit["image_parts"]:
        print("PDF 쪽:", image["pdf_page_number"], "| 그림 이름:", image["file_name"])
        for link in image["descriptions"]:
            print("그림 설명:", link["description"], "| 연결 검토 상태:", link["linkage_status"])
        url = image["public_url"]
        if url and url not in shown_urls:
            shown_urls.add(url)
            # width는 화면 표시 크기이며 원본 파일의 크기를 바꾸지 않습니다.
            display(Image(url=url, width=700))
        elif not url:
            print("공개 그림 주소가 없습니다. SUPABASE_URL과 업로드 상태를 확인하세요.")


## 이번에 실제 실행한 결과 — 2026-10-02

Python 실행 파일 `manual.py db-search`에서 세 질문을 한 실행으로 조회했습니다.

| 질문 | 첫 결과의 연결 근거 | 매뉴얼 쪽 | 그림 | 코사인 유사도 |
|---|---|---|---|---|
| 엔진 오일의 용량과 추천 사양을 알려줘. | PDF 43쪽 표 + 각주·주의사항 | 38 | 0 | 0.850 |
| 엔진 오일의 SAE 점도는 무엇인가요? | PDF 44쪽 점도 분류표 | 39 | 2 | 0.713 |
| 차대번호는 어디에서 확인하나요? | PDF 45쪽 차대번호 | 40 | 1 | 0.607 |

첫 질문은 각주 청크가 가장 높게 검색됐으며 연결된 표도 함께 반환됐습니다. 모델 준비 구간 약 33.6초, 질문 임베딩·DB 조회는 각각 약 2.4/1.4/1.5초였습니다. 실행 환경에 따라 달라집니다.

기존 전체 실험의 글자 검색·질문 목적 규칙은 이 DB 검색에 아직 결합하지 않았습니다. 위 세 사례의 검색 연결 확인이며, 전체 검색 품질이나 챗봇 답변 정확도를 보장하지 않습니다. 근거 부족 판정과 답변 생성은 이후 단계입니다.

**코드 읽는 순서:** `manual.py`의 db-search → `db_search_service.py`의 search → `mappers/manual_store.sql.txt`의 search_chunks.
